# Loading data

In [ ]:
from google.colab import drive
import pickle
import pandas as pd

In [ ]:
with open('covid_flow.pkl', 'rb') as f:
    data = pickle.load(f)

print(data)

        t_point  омепразол_stat_control  надропарин кальция_stat_control  \
case                                                                       
GACAk+Q     t_0                       1                                1   
GACAmDB     t_0                       1                                1   
GACAmDG     t_0                       1                                0   
GACAlшш     t_0                       0                                1   
GACAjу.     t_0                       0                                1   
...         ...                     ...                              ...   
GACBAk=     t_1                       0                                0   
GACBAy[     t_1                       0                                0   
GACBAyВ     t_1                       1                                0   
GACBAИ9     t_4                       1                                0   
GACBAХп     t_1                       0                                0   

         эз

/tmp/ipykernel_9923/2409083345.py:2: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  data = pickle.load(f)


#Calculating the hash

In [ ]:
import hashlib
import json
import os

In [ ]:
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def cal_hash_file(f_path, block_size=65536):
  sha256 = hashlib.sha256()
  try:
    with open(f_path, "rb") as f:
      for block in iter(lambda: f.read(block_size), b""):
        sha256.update(block)
    return sha256.hexdigest()
  except Exception as e:
    return f"Error:{e}"


In [ ]:
def g_drive_folder(fol_path, output_json="manf.json"):
  if not os.path.exists(fol_path):
    print(f"Error {fol_path} doesnt exsist")
    return

  manifest = {}
  output_json = os.path.join(fol_path, output_json)
  print(f"... ... ... ... {fol_path}")

  for root, _, files in os.walk(fol_path):
    for file in files:
      if file == output_json or file.startswith("._"):
        continue
      full_path = os.path.join(root,file)
      rel_path = os.path.relpath(full_path, fol_path)
      file_hash=cal_hash_file(full_path)
      manifest[rel_path]=file_hash

  with open(output_json, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4, ensure_ascii=False)

  print(f"Files successfully committed '{len(manifest)}'")
  print(f"Save : {output_json}")

In [ ]:
T_folder = "/content/drive/MyDrive/COVID/Version"

In [ ]:
g_drive_folder(T_folder)

... ... ... ... /content/drive/MyDrive/COVID/Version
Files successfully committed 1
Save : /content/drive/MyDrive/COVID/Version/manf.json


#Checking hash

In [ ]:
def checking_gdrive_fol(fol_path, manifest_name="manf.json"):
  manifest_path = os.path.join(fol_path, manifest_name)
  if not os.path.exists(manifest_path):
    print(f"Error '{manifest_name}' doesnt exsist")
    return False
  with open(manifest_path, "r", encoding="utf-8") as f:
    saved_man = json.load(f)

  current_files=set()
  modified_files = []
  corrupted_files=[]

  print(f"... ... {fol_path}")
  for root, _, files in os.walk(fol_path):
    for file in files:
      if file == manifest_name or file.startswith("._"):
        continue

      full_path= os.path.join(root, file)
      rel_path=os.path.relpath(full_path, fol_path)
      current_files.add(rel_path)

      current_hash = cal_hash_file(full_path)

      if rel_path not in saved_man:
        print(f"New file {rel_path}")
      elif current_hash != saved_man[rel_path]:
        if current_hash.startswith("Error"):
          corrupted_files.append(rel_path)
          print(f"Error read {rel_path}")
        else:
          modified_files.append(rel_path)
          print(f" Changed {rel_path} ")
  deleted_files = [f for f in saved_man if f not in current_files]
  for f in deleted_files:
    print(f" File has deleted from fol {f}")

  print("\n"+"="*25)
  if not modified_files and not deleted_files and not corrupted_files:
    print("Evth is ok")
    return True
  else:
    print(f" -Changed: {len(modified_files)}")
    print(f" -Deleted: {len(deleted_files)}")
    print(f" -Error reading {len(corrupted_files)}")
    return False


In [ ]:
T_fol = "/content/drive/MyDrive/COVID/Version"

In [ ]:
is_safe = checking_gdrive_fol(T_fol)

... ... /content/drive/MyDrive/COVID/Version

Evth is ok


#Checking the dataset structure

In [ ]:
errors_found = False

1

In [ ]:
actual_col = data.shape[1]
expected_col = 54

if actual_col == expected_col:
  print(f"54 is correct")
else:
  errors_found = True
  print(f" Error, not {expected_col} is {actual_col}")


54 is correct


2

In [ ]:
col_case = "case"
col_t_point = "t_point"

if col_case not in data.columns or col_t_point not in data.columns:
  print(f"columns {col_case} or {col_t_point} not founded")
  errors_found = True
else:
  duplicates = data[data.duplicated(subset=[col_case, col_t_point], keep=False)]

  if duplicates.empty:
    print(f"Uniqueness ERROR: Non-unique pairs found [{col_case}+{col_t_point}]")
    print(f" - Total rows with duplicates {duplicates.shape[0]} first 5:")
    print(
        duplicates[[col_case, col_t_point]]
        .sort_values(be=[col_case, col_t_point])
        .head(5)
    )

columns case or t_point not founded


3